importing modules

In [1]:
import pandas as pd
import numpy as np

Basic Describe and Null Check of both dataset

In [ ]:
district = pd.read_csv("./datasets/nepal_district_centroids.csv")
dengu_temporal = pd.read_csv("./datasets/dengu_temporal.csv")

print(f"Length of District Data: {len(district)}")
print("------------------------------------")
print("District Data Described: ")
display(district.describe())
print("------------------------------------")
print("District Data Null Vals")
display(district.isnull().sum())

print("\n------------------------------------\n------------------------------------\n")

print(f"Length of Temporal Data: {len(dengu_temporal)}")
print("------------------------------------")
print("Temporal Data Described: ")
display(dengu_temporal.describe())
print("------------------------------------")
print("Temporal Data Null Vals")
display(dengu_temporal.isnull().sum())


In [ ]:
print("Temporal Data")
display(dengu_temporal.dtypes)

In [ ]:
# Print first 5 data of temporal
print('Temporal Data structure')
dengu_temporal.head()

Show Different value counts in the Temporal Data

In [ ]:
display(dengu_temporal["case_definition_standardised"].value_counts())
display(dengu_temporal["T_res"].value_counts())

In [ ]:
# remove Yearly and weekly dataset as they carry 3% only data in whole dataset
dengu_temporal = dengu_temporal[(dengu_temporal["T_res"]!="Year") & (dengu_temporal["T_res"]!="Week")]

Add new Location Column and remove unnecessary

In [ ]:
# Location In Dataset is Added 
dengu_temporal["Location"] = dengu_temporal["adm_2_name"].fillna(dengu_temporal["full_name"].str.split(",").str[-1])

# Remove unnecessary Columns
dengu_temporal = dengu_temporal.drop(
    columns="adm_0_name,adm_1_name,adm_2_name,full_name,ISO_A0,FAO_GAUL_code,RNE_iso_code,IBGE_code,S_res,region,UUID,case_definition_standardised,T_res".split(",")
)

dengu_temporal[:10]

Extract month using start date. Then delete other unnecessary dates

In [ ]:
# convert to date time format
dengu_temporal["calendar_start_date"] = pd.to_datetime(dengu_temporal["calendar_start_date"])

#month map
month_map = {
    1: "Jan", 2: "Feb", 3: "Mar", 4: "Apr", 
    5: "May", 6: "Jun", 7: "Jul", 8: "Aug", 
    9: "Sep", 10: "Oct", 11: "Nov", 12: "Dec"
}

dengu_temporal["Month"] = dengu_temporal["calendar_start_date"].dt.month.map(month_map)
dengu_temporal.head()

In [ ]:
# show all location counts
display(dengu_temporal["Location"].value_counts())
print(f'Total Length = {len(dengu_temporal)}')
dengu_temporal.head()

The Location Nepal can give specific location and it only contains 3.5% of total dataset we delete rows

In [ ]:
dengu_temporal = dengu_temporal[dengu_temporal["Location"]!="NEPAL"]
dengu_temporal.head()

Merge latitude and longitude data to dengu data

In [ ]:
merged_df = pd.merge(dengu_temporal, district, left_on="Location", right_on="DISTRICT", how="left")

print(f"Length of Merged df = {len(merged_df)}")
merged_df.head()

In [ ]:
merged_df.drop(columns=["DISTRICT"], inplace=True)
merged_df.head()

In [ ]:
print(merged_df["LATITUDE"].isna().sum())
print(merged_df["LONGITUDE"].isna().sum())
print(len(merged_df))

In [ ]:
merged_df = merged_df.dropna(subset=["LATITUDE", "LONGITUDE"])

merged_df["calendar_start_date"] = pd.to_datetime(
    merged_df["calendar_start_date"],
    errors="coerce"
)

merged_df["calendar_end_date"] = pd.to_datetime(
    merged_df["calendar_end_date"],
    errors="coerce"
)

print(merged_df[["calendar_start_date", "calendar_end_date"]].dtypes)

Referenced from and used from https://open-meteo.com/en/docs/historical-weather-api. Modified to function.

In [ ]:
import openmeteo_requests
import pandas as pd

# Initialize Open-Meteo client
openmeteo = openmeteo_requests.Client()


def get_weather_summary(start_dates, end_dates, latitudes, longitudes):
    # Open-Meteo archive API
    url = "https://archive-api.open-meteo.com/v1/archive"

    # Store results
    results = []

    i=1
    # Iterate through each location/date range
    for start, end, lat, lon in zip(start_dates, end_dates, latitudes, longitudes):
        
        start = pd.to_datetime(start).strftime("%Y-%m-%d")
        end = pd.to_datetime(end).strftime("%Y-%m-%d")

        # Request parameters
        params = {
            "latitude": lat,
            "longitude": lon,
            "start_date": start,
            "end_date": end,
            "daily": [
                "temperature_2m_mean",
                "rain_sum",
                "relative_humidity_2m_mean",
                "soil_moisture_0_to_100cm_mean",
                "soil_temperature_0_to_100cm_mean",
                "snowfall_sum",
                "precipitation_sum",
            ],
        }

        # Send request
        response = openmeteo.weather_api(url, params=params)[0]
        daily = response.Daily()

        # Save the average values for this request
        results.append({
            "latitude": lat,
            "longitude": lon,
            "start_date": start,
            "end_date": end,
            "monthly_avg_temperature": daily.Variables(0).ValuesAsNumpy().mean(),
            "avg_daily_rain": daily.Variables(1).ValuesAsNumpy().mean(),
            "avg_daily_humidity": daily.Variables(2).ValuesAsNumpy().mean(),
            "avg_daily_soil_moisture": daily.Variables(3).ValuesAsNumpy().mean(),
            "avg_daily_soil_temperature": daily.Variables(4).ValuesAsNumpy().mean(),
            "avg_daily_snowfall": daily.Variables(5).ValuesAsNumpy().mean(),
            "avg_daily_precipitation": daily.Variables(6).ValuesAsNumpy().mean(),
        })
        
        print(f"completed {i} ....")
        i+=1

    # Return one row per input
    return pd.DataFrame(results)

In [ ]:
weather_data = get_weather_summary(
    merged_df["calendar_start_date"],
    merged_df["calendar_end_date"],
    merged_df["LATITUDE"],
    merged_df["LONGITUDE"]
)

In [ ]:
final_df = pd.merge(
    merged_df,
    weather_data,
    left_on=[
        "LATITUDE",
        "LONGITUDE",
    ],
    right_on=["latitude", "longitude",],
    how="left",
)

print("\nDone!")
final_df.head()

In [ ]:
final_df.drop(columns=["calendar_start_date", "calendar_end_date", "longitude", "latitude", "LONGITUDE", "LATITUDE", "start_date", "end_date"], inplace=True)

In [ ]:
len(final_df.columns)

In [ ]:
final_df.head()

In [ ]:
# rainfall into classification

final_df["daily_rain_density"] = np.where(
    final_df["avg_daily_rain"] == 0, "no",
    np.where(
        final_df["avg_daily_rain"] <= 2.5, "low",
        np.where(
            final_df["avg_daily_rain"] <= 10, "moderate",
            np.where(
                final_df["avg_daily_rain"] <= 35, "high",
                "extreme"
            )
        )
    )
)


# humidity into classification

final_df["average_humidity"] = np.where(
    final_df["avg_daily_humidity"] < 40, "low",
    np.where(
        final_df["avg_daily_humidity"] < 60, "moderate",
        np.where(
            final_df["avg_daily_humidity"] < 80, "high",
            "extreme"
        )
    )
)


# avg_daily_soil_moisture into classification

final_df["avg_soil_moisture"] = np.where(
    final_df["avg_daily_soil_moisture"] < 0.15, "low",
    np.where(
        final_df["avg_daily_soil_moisture"] < 0.30, "moderate",
        np.where(
            final_df["avg_daily_soil_moisture"] < 0.45, "high",
            "extreme"
        )
    )
)


# avg_daily_soil_temperature into classification

final_df["avg_soil_temperature"] = np.where(
    final_df["avg_daily_soil_temperature"] < 15, "low",
    np.where(
        final_df["avg_daily_soil_temperature"] < 25, "moderate",
        np.where(
            final_df["avg_daily_soil_temperature"] < 35, "high",
            "extreme"
        )
    )
)


# avg_daily_snowfall into classification

final_df["avg_snowfall"] = np.where(
    final_df["avg_daily_snowfall"] == 0, "none",
    np.where(
        final_df["avg_daily_snowfall"] <= 5, "light",
        np.where(
            final_df["avg_daily_snowfall"] <= 20, "moderate",
            "heavy"
        )
    )
)


# avg_daily_precipitation into classification

final_df["avg_precipitation"] = np.where(
    final_df["avg_daily_precipitation"] == 0, "no",
    np.where(
        final_df["avg_daily_precipitation"] <= 2.5, "low",
        np.where(
            final_df["avg_daily_precipitation"] <= 10, "moderate",
            np.where(
                final_df["avg_daily_precipitation"] <= 35, "high",
                "extreme"
            )
        )
    )
)

In [ ]:
len(final_df.columns)

In [ ]:
final_df.head()

In [ ]:
final_df = final_df.sample(frac=1, random_state=42, ignore_index=True)
final_df.head()

In [10]:
final_df.to_csv("../datasets/dengu_predictions_in_district_final.csv", index=False)

Train Val test split 70%, 15%, 15%

In [11]:
final_data = pd.read_csv("../datasets/dengu_predictions_in_district_final.csv")
final_data.head()

,Year,dengue_total,Location,Month,monthly_avg_temperature,avg_daily_rain,avg_daily_humidity,avg_daily_soil_moisture,avg_daily_soil_temperature,avg_daily_snowfall,avg_daily_precipitation,daily_rain_density,average_humidity,avg_soil_moisture,avg_soil_temperature,avg_snowfall,avg_precipitation
0,2022,1,RUPANDEHI,Apr,27.541040,0.140000,40.643658,0.180724,25.653076,0.00,0.140000,low,moderate,moderate,high,none,low
1,2024,0,SURKHET,Jan,21.709210,2.438710,54.626945,0.282942,18.936808,0.00,2.438710,low,moderate,moderate,moderate,none,low
2,2022,0,RASUWA,Jul,-10.367068,0.000000,47.298428,0.323489,-2.997343,1.53,2.171428,no,moderate,high,low,light,low
3,2022,0,BAITADI,May,20.027569,4.980001,85.651030,0.473050,21.403994,0.00,4.980001,moderate,extreme,extreme,moderate,none,moderate
4,2024,4,KANCHANPUR,Feb,13.511224,0.216129,79.257805,0.300749,17.036068,0.00,0.216129,low,high,high,moderate,none,low


In [14]:
df = final_data.sample(frac=1, random_state=42).reset_index(drop=True)
df.head()

,Year,dengue_total,Location,Month,monthly_avg_temperature,avg_daily_rain,avg_daily_humidity,avg_daily_soil_moisture,avg_daily_soil_temperature,avg_daily_snowfall,avg_daily_precipitation,daily_rain_density,average_humidity,avg_soil_moisture,avg_soil_temperature,avg_snowfall,avg_precipitation
0,2024,0,KALIKOT,May,10.223880,19.783870,88.951584,0.418761,10.755308,0.000000,19.783870,high,extreme,high,low,none,high
1,2022,0,PARSA,Feb,30.428612,6.673333,62.701965,0.234122,28.814114,0.000000,6.673333,moderate,high,moderate,high,none,moderate
2,2024,19,SALYAN,Sep,10.010685,0.100000,70.479485,0.319492,11.226091,0.000000,0.100000,low,high,high,low,none,low
3,2022,1,BAJHANG,Jan,-6.802581,0.000000,34.566444,0.366226,-1.123817,0.388387,0.551613,no,low,high,low,light,low
4,2022,0,KALIKOT,May,10.058961,13.132258,89.504610,0.415793,10.209129,0.000000,13.132258,high,extreme,high,low,none,high


In [15]:
n = len(df)

# Split sizes
train_size = int(0.7 * n)
val_size = int(0.15 * n)

print(train_size)
print(val_size)

62596
13413


In [16]:
train_df = df.iloc[:train_size]
val_df = df.iloc[train_size:train_size + val_size]
test_df = df.iloc[train_size + val_size:]

In [17]:
print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Train: (62596, 17)
Validation: (13413, 17)
Test: (13415, 17)


In [18]:
train_df.to_csv("../datasets/train.csv", index=False)
test_df.to_csv("../datasets/test.csv", index=False)
val_df.to_csv("../datasets/val.csv", index=False)